In [11]:
import pandas as pd
from langcodes import Language
import uuid
from datasets import load_dataset, Dataset

In [3]:
seed_df = load_dataset("ljvmiranda921/msde-seed-S1", split="train").to_pandas()
print(seed_df.columns)

Generating train split: 100%|██████████| 449017/449017 [00:00<00:00, 691487.96 examples/s]


Index(['id', 'source', 'language', 'strategy', 'source_id', 'prompt',
       'response'],
      dtype='object')


In [4]:
LANG_MAPPING = {"Tagalog": "tl", "Filipino": "fil"}

Process Cohere Aya Collection

In [8]:
def _iso3_to_iso2(code: str) -> str:
    tag = Language.get(code).to_tag()
    iso2 = tag.split("-")[0]
    return iso2

In [12]:
aya_ds = load_dataset("CohereLabs/aya_collection", "aya_dataset", split="train")
filtered_df = aya_ds.to_pandas()
filtered_df["language"] = filtered_df["language"].apply(_iso3_to_iso2)
filtered_df = filtered_df[filtered_df["language"].isin(set(LANG_MAPPING.values()))].reset_index(drop=True)  # fmt: skip
aya_df = pd.DataFrame(
    {
        "id": [uuid.uuid4().hex for _ in range(len(filtered_df))],
        "source": "CohereLabs/aya_collection",
        "language": filtered_df["language"].astype(str).values,
        "prompt": filtered_df["inputs"].astype(str).values,
        "response": filtered_df["targets"].astype(str).values,
        "strategy": [["generate", "respond"] for _ in range(len(filtered_df))],
        "source_id": filtered_df["id"].astype(str).values,
    }
)

In [35]:
len(aya_df)

1241

Process AllenAI Wildchat

In [36]:
wildchat_df_raw = pd.read_json("data/tl.jsonl.zst", lines=True, compression="zstd")

In [42]:
wildchat_df = pd.DataFrame(
    {
        # fmt: off
        "id": [uuid.uuid4().hex for _ in range(len(wildchat_df_raw))],
        "source": wildchat_df_raw["source"].astype(str).values,
        "language": "tl",
        "prompt": [row[0]["value"] for _, row in wildchat_df_raw.conversations.items()],
        "response": [row[1]["value"] for _, row in wildchat_df_raw.conversations.items()],
        "strategy": [["generate", "respond"] for _ in range(len(wildchat_df_raw))],
        "source_id": "agentlans/allenai-WildChat_" + wildchat_df_raw.index.astype(str),
        # fmt: on
    }
)

In [ ]:
len(wildchat_df)

,id,source,language,prompt,response,strategy,source_id
0,4a1f9bbbec8046abb9cba87938f8b78c,allenai/WildChat-4.8M,tl,Respond to this message in the appropriate lan...,Ang pag-ibig ay isang napaka-komplikadong emos...,"[generate, respond]",agentlans/allenai-WildChat_0
1,79d3d700871a4ea7af999710c06f32ba,allenai/WildChat-4.8M,tl,anong AI model ka?\nspeak in a more casual tag...,Hey! Ako yung AI assistant mo na laging ready ...,"[generate, respond]",agentlans/allenai-WildChat_1
2,3b7dd25fc6a34054a4688768a3d382be,allenai/WildChat-4.8M,tl,"Ok, sa problem na ito, ang lumalabas na result...",Magandang araw! Mukhang mayroong pagkalito sa ...,"[generate, respond]",agentlans/allenai-WildChat_2
3,d40ae36dbad845149b24d83dbdadf430,allenai/WildChat-4.8M,tl,Ipaliwanag ng maikli kaugnay sa Bana na Espiri...,"Ang ""Bana na Espiritu"" ay isang konsepto na tu...","[generate, respond]",agentlans/allenai-WildChat_3
4,827cba1a9198483697074f4e151f10da,allenai/WildChat-4.8M,tl,Summarize this text in Filipino (make sure to ...,"Ang tekstong ""Ang Kahalagahan ng Pangangalaga ...","[generate, respond]",agentlans/allenai-WildChat_4
...,...,...,...,...,...,...,...
1242,a32a1e43b74043ce9b8116ef8f134527,allenai/WildChat-4.8M,tl,User: bakit mahalaga ang buhay ng tao?\nAssist...,Mahalaga ang buhay ng tao dahil ang bawat indi...,"[generate, respond]",agentlans/allenai-WildChat_1242
1243,ce500183c3bf48fa9972c26dacbd45c2,allenai/WildChat-4.8M,tl,Respond to this message in the appropriate lan...,Ang aking pilosopiya sa edukasyon ay tungkol s...,"[generate, respond]",agentlans/allenai-WildChat_1243
1244,6b9d3c34e67849d5955b6f126390f923,allenai/WildChat-4.8M,tl,Respond to this message in the appropriate lan...,"Pasensya na, pero wala akong kakayahan na mag-...","[generate, respond]",agentlans/allenai-WildChat_1244
1245,2b20144386fe4e788d88f454dc0ffc0b,allenai/WildChat-1M,tl,Wastong pagpapaliwanag ng cycle graph na estra...,Ang cycle graph na estratehiya sa pagtuturo ng...,"[generate, respond]",agentlans/allenai-WildChat_1245
